# Model Baselines

This notebook applies logistic regression, random forest, and XGBoost to try and get a baseline for predictions.

In [1]:
%pip install tensorflow scikit-learn matplotlib keras-tuner tensorboard xgboost


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
import keras_tuner as kt
from xgboost import XGBClassifier
from core.data import get_logistic_regression_results, get_model_data, get_naive_baseline_results, get_ticker_features, get_train_test_data

## Create a list of the tickers to use

In [3]:

# Create arrays of tickers from different sectors
TECHNOLOGY_AND_COMMUNICATION_SERVICES_TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'GOOG', 'AMZN', 'META', 'NVDA', 'TSLA', 'AVGO', 'ORCL',
    'CRM', 'ADBE', 'AMD', 'INTC', 'CSCO', 'IBM', 'QCOM', 'TXN', 'NOW', 'INTU', 'ACN',
    'DIS', 'NFLX', 'CMCSA', 'TMUS', 'VZ', 'T',
]

FINANCIALS_TICKERS = [
    'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'BLK', 'AXP', 'V', 'MA',
    'SPGI', 'CB', 'PGR',
]

HEALTHCARE_TICKERS = [
    'UNH', 'JNJ', 'LLY', 'MRK', 'ABBV', 'PFE', 'TMO', 'ABT', 'DHR', 'AMGN',
    'GILD', 'BMY', 'ISRG', 'MDT',
]

CONSUMER_STAPLES_AND_DISCRETIONARY_TICKERS = [
    'PG', 'KO', 'PEP', 'COST', 'WMT', 'HD', 'MCD', 'NKE', 'SBUX', 'LOW',
    'TGT', 'CL', 'EL',
]

INDUSTRIALS_TICKERS = [
    'CAT', 'DE', 'GE', 'HON', 'UPS', 'RTX', 'LMT', 'BA', 'MMM', 'ETN', 'EMR',
]

ENERGY_TICKERS = [
    'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'MPC', 'PSX',
]

UTILITIES_TICKERS = [
    'NEE', 'SO', 'DUK', 'AEP', 'EXC', 'SRE',
]

MATERIALS_TICKERS = [
    'LIN', 'APD', 'SHW', 'FCX', 'NEM',
]

REAL_ESTATE_TICKERS = [
    'AMT', 'PLD', 'EQIX',
]

TICKER_GROUPS = {
    'Technology and communication services': TECHNOLOGY_AND_COMMUNICATION_SERVICES_TICKERS,
    'Financials': FINANCIALS_TICKERS,
    'Healthcare': HEALTHCARE_TICKERS,
    'Consumer staples and discretionary': CONSUMER_STAPLES_AND_DISCRETIONARY_TICKERS,
    'Industrials': INDUSTRIALS_TICKERS,
    'Energy': ENERGY_TICKERS,
    'Utilities': UTILITIES_TICKERS,
    'Materials': MATERIALS_TICKERS,
    'Real estate': REAL_ESTATE_TICKERS,
}

TICKERS = [ticker for group in TICKER_GROUPS.values() for ticker in group]

In [4]:
TEST_TICKER = 'AAPL'
TRAIN_START = '2015-01-01'
TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'
TEST_END = '2024-12-31'


In [5]:
ticker_features = {}
for ticker in TICKERS:
    df = get_ticker_features(ticker, start=TRAIN_START, end=TEST_END)
    print(f'{ticker}: {len(df)} rows  ({df.index[0].date()} to {df.index[-1].date()})')
    ticker_features[ticker] = df

print(f'Loaded the {len(ticker_features)} tickers requested.')

AAPL: 2482 rows  (2015-02-20 to 2024-12-30)
MSFT: 2482 rows  (2015-02-20 to 2024-12-30)
GOOGL: 2482 rows  (2015-02-20 to 2024-12-30)
GOOG: 2482 rows  (2015-02-20 to 2024-12-30)
AMZN: 2482 rows  (2015-02-20 to 2024-12-30)
META: 2482 rows  (2015-02-20 to 2024-12-30)
NVDA: 2482 rows  (2015-02-20 to 2024-12-30)
TSLA: 2482 rows  (2015-02-20 to 2024-12-30)
AVGO: 2482 rows  (2015-02-20 to 2024-12-30)
ORCL: 2482 rows  (2015-02-20 to 2024-12-30)
CRM: 2482 rows  (2015-02-20 to 2024-12-30)
ADBE: 2482 rows  (2015-02-20 to 2024-12-30)
AMD: 2482 rows  (2015-02-20 to 2024-12-30)
INTC: 2482 rows  (2015-02-20 to 2024-12-30)
CSCO: 2482 rows  (2015-02-20 to 2024-12-30)
IBM: 2482 rows  (2015-02-20 to 2024-12-30)
QCOM: 2482 rows  (2015-02-20 to 2024-12-30)
TXN: 2482 rows  (2015-02-20 to 2024-12-30)
NOW: 2482 rows  (2015-02-20 to 2024-12-30)
INTU: 2482 rows  (2015-02-20 to 2024-12-30)
ACN: 2482 rows  (2015-02-20 to 2024-12-30)
DIS: 2482 rows  (2015-02-20 to 2024-12-30)
NFLX: 2482 rows  (2015-02-20 to 2024-1

## Select Features for Training

In [6]:
ticker_features[TEST_TICKER].columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'Dividends',
 'Stock Splits',
 'rsi',
 'macd',
 'macd_signal',
 'bb_upper',
 'bb_mid',
 'bb_lower',
 'atr',
 'obv',
 'hlc3',
 'rsi_scaled',
 'target_close',
 'Open_return',
 'High_return',
 'Low_return',
 'Close_return',
 'Volume_return',
 'hlc3_return',
 'volume_ratio',
 'volume_ratio_clipped',
 'obv_change_to_avg_volume',
 'macd_to_close',
 'macd_signal_to_close',
 'atr_to_close',
 'bb_upper_to_close',
 'bb_mid_to_close',
 'bb_lower_to_close',
 'market_close',
 'growth_1d',
 'market_growth_1d',
 'excess_growth_vs_market_1d',
 'beat_market_1d',
 'growth_2d',
 'market_growth_2d',
 'excess_growth_vs_market_2d',
 'beat_market_2d',
 'growth_3d',
 'market_growth_3d',
 'excess_growth_vs_market_3d',
 'beat_market_3d',
 'growth_5d',
 'market_growth_5d',
 'excess_growth_vs_market_5d',
 'beat_market_5d',
 'growth_10d',
 'market_growth_10d',
 'excess_growth_vs_market_10d',
 'beat_market_10d',
 'growth_20d',
 'market_growth_20d',
 'excess_growth_vs_

In [7]:
LOGISTIC_BASELINE_FEATURES = [
    # Daily stock movement
    'Open_return',
    'High_return',
    'Low_return',
    'Close_return',
    'Volume_return',
    'hlc3_return',

    # Bounded / normalized indicators
    'rsi',
    'volume_ratio_clipped',
    'obv_change_to_avg_volume',
    'macd_to_close',
    'macd_signal_to_close',
    'atr_to_close',
    'bb_upper_to_close',
    'bb_mid_to_close',
    'bb_lower_to_close',
    'rsi_scaled',

    # Recent stock growth
    'growth_1d',
    'growth_2d',
    'growth_3d',
    'growth_5d',
    'growth_10d',
    'growth_20d',

    # Market-relative growth
    'market_growth_1d',
    'market_growth_2d',
    'market_growth_3d',
    'market_growth_5d',
    'market_growth_10d',
    'market_growth_20d',

    'excess_growth_vs_market_1d',
    'excess_growth_vs_market_2d',
    'excess_growth_vs_market_3d',
    'excess_growth_vs_market_5d',
    'excess_growth_vs_market_10d',
    'excess_growth_vs_market_20d',
]

## Logistic regression: 1-day up model

This baseline trains one pooled logistic regression model to estimate the probability that a stock closes higher one trading day later. The target uses `target_close.shift(-1)`, while the input features are trailing/current-day features.


In [8]:
FORECAST_HORIZON = 1
TARGET_COL = f'target_up_{FORECAST_HORIZON}d'

model_data = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, FORECAST_HORIZON)

train_rows, test_rows, scaler, X_train, X_test, y_train, y_test = get_train_test_data(
    model_data,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL,
    TRAIN_END,
    TEST_START,
)

print(f'Training samples: {len(y_train)}')
print(f'Test samples: {len(y_test)}')
print(f'Test up rate: {y_test.mean():.2%}')


Training samples: 220869
Test samples: 24750
Test up rate: 52.55%


## Naive majority baseline

This baseline always predicts the most common class in the test set. Any trained model should beat this accuracy to be useful.


In [9]:
naive_baseline_results = get_naive_baseline_results(y_test, FORECAST_HORIZON)
metric_cols = ['up_rate', 'accuracy', 'log_loss', 'auc']
naive_baseline_results


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,1d,24750,0.5255,1,0.5255,0.6918,0.5


## Logistic regression baseline

This trains a pooled logistic regression model across all tickers and compares it with the naive majority baseline.


In [10]:
logistic_1d_model, logistic_1d_results = get_logistic_regression_results(
    X_train,
    y_train,
    X_test,
    y_test,
    FORECAST_HORIZON,
)

pd.concat([naive_baseline_results, logistic_1d_results], ignore_index=True)


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,1d,24750,0.5255,1.0,0.5255,0.6918,0.500
1,Logistic regression,1d,24750,0.5255,NaN,0.5253,0.6922,0.495


## 2-Day Forecast Window

In [11]:
TARGET_COL_2D = 'target_up_2d'

model_data_2d = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, 2)

train_rows_2d, test_rows_2d, scaler_2d, X_train_2d, X_test_2d, y_train_2d, y_test_2d = get_train_test_data(
    model_data_2d,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL_2D,
    TRAIN_END,
    TEST_START,
)

print(f'2-day training samples: {len(y_train_2d)}')
print(f'2-day test samples: {len(y_test_2d)}')
print(f'2-day test up rate: {y_test_2d.mean():.2%}')

naive_baseline_results_2d = get_naive_baseline_results(y_test_2d, 2)
logistic_2d_model, logistic_2d_results = get_logistic_regression_results(
    X_train_2d,
    y_train_2d,
    X_test_2d,
    y_test_2d,
    2,
)

pd.concat([naive_baseline_results_2d, logistic_2d_results], ignore_index=True)


2-day training samples: 220869
2-day test samples: 24651
2-day test up rate: 53.69%


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,2d,24651,0.5369,1.0,0.5369,0.6904,0.5000
1,Logistic regression,2d,24651,0.5369,NaN,0.5302,0.6914,0.4942


## 3-Day Forecast Window

In [12]:
TARGET_COL_3D = 'target_up_3d'

model_data_3d = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, 3)

train_rows_3d, test_rows_3d, scaler_3d, X_train_3d, X_test_3d, y_train_3d, y_test_3d = get_train_test_data(
    model_data_3d,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL_3D,
    TRAIN_END,
    TEST_START,
)

print(f'3-day training samples: {len(y_train_3d)}')
print(f'3-day test samples: {len(y_test_3d)}')
print(f'3-day test up rate: {y_test_3d.mean():.2%}')

naive_baseline_results_3d = get_naive_baseline_results(y_test_3d, 3)
logistic_3d_model, logistic_3d_results = get_logistic_regression_results(
    X_train_3d,
    y_train_3d,
    X_test_3d,
    y_test_3d,
    3,
)

pd.concat([naive_baseline_results_3d, logistic_3d_results], ignore_index=True)


3-day training samples: 220869
3-day test samples: 24552
3-day test up rate: 54.27%


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,3d,24552,0.5427,1.0,0.5427,0.6895,0.5000
1,Logistic regression,3d,24552,0.5427,NaN,0.5376,0.6903,0.4999


## 4-Day Forecast Window

In [13]:
TARGET_COL_4D = 'target_up_4d'

model_data_4d = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, 4)

train_rows_4d, test_rows_4d, scaler_4d, X_train_4d, X_test_4d, y_train_4d, y_test_4d = get_train_test_data(
    model_data_4d,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL_4D,
    TRAIN_END,
    TEST_START,
)

print(f'4-day training samples: {len(y_train_4d)}')
print(f'4-day test samples: {len(y_test_4d)}')
print(f'4-day test up rate: {y_test_4d.mean():.2%}')

naive_baseline_results_4d = get_naive_baseline_results(y_test_4d, 4)
logistic_4d_model, logistic_4d_results = get_logistic_regression_results(
    X_train_4d,
    y_train_4d,
    X_test_4d,
    y_test_4d,
    4,
)

pd.concat([naive_baseline_results_4d, logistic_4d_results], ignore_index=True)


4-day training samples: 220869
4-day test samples: 24453
4-day test up rate: 54.53%


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,4d,24453,0.5453,1.0,0.5453,0.689,0.5000
1,Logistic regression,4d,24453,0.5453,NaN,0.5424,0.690,0.4952


## 10-Day Forecast Window

In [14]:
TARGET_COL_10D = 'target_up_10d'

model_data_10d = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, 10)

train_rows_10d, test_rows_10d, scaler_10d, X_train_10d, X_test_10d, y_train_10d, y_test_10d = get_train_test_data(
    model_data_10d,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL_10D,
    TRAIN_END,
    TEST_START,
)

print(f'10-day training samples: {len(y_train_10d)}')
print(f'10-day test samples: {len(y_test_10d)}')
print(f'10-day test up rate: {y_test_10d.mean():.2%}')

naive_baseline_results_10d = get_naive_baseline_results(y_test_10d, 10)
logistic_10d_model, logistic_10d_results = get_logistic_regression_results(
    X_train_10d,
    y_train_10d,
    X_test_10d,
    y_test_10d,
    10,
)

pd.concat([naive_baseline_results_10d, logistic_10d_results], ignore_index=True)


10-day training samples: 220869
10-day test samples: 23859
10-day test up rate: 56.47%


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,10d,23859,0.5647,1.0,0.5647,0.6848,0.5000
1,Logistic regression,10d,23859,0.5647,NaN,0.5658,0.6847,0.5133


## 20-Day Forecast Window

In [15]:
TARGET_COL_20D = 'target_up_20d'

model_data_20d = get_model_data(ticker_features, LOGISTIC_BASELINE_FEATURES, 20)

train_rows_20d, test_rows_20d, scaler_20d, X_train_20d, X_test_20d, y_train_20d, y_test_20d = get_train_test_data(
    model_data_20d,
    LOGISTIC_BASELINE_FEATURES,
    TARGET_COL_20D,
    TRAIN_END,
    TEST_START,
)

print(f'20-day training samples: {len(y_train_20d)}')
print(f'20-day test samples: {len(y_test_20d)}')
print(f'20-day test up rate: {y_test_20d.mean():.2%}')

naive_baseline_results_20d = get_naive_baseline_results(y_test_20d, 20)
logistic_20d_model, logistic_20d_results = get_logistic_regression_results(
    X_train_20d,
    y_train_20d,
    X_test_20d,
    y_test_20d,
    20,
)

pd.concat([naive_baseline_results_20d, logistic_20d_results], ignore_index=True)


20-day training samples: 220869
20-day test samples: 22869
20-day test up rate: 59.53%


,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc
0,Naive majority baseline,20d,22869,0.5953,1.0,0.5953,0.6749,0.5000
1,Logistic regression,20d,22869,0.5953,NaN,0.5952,0.6757,0.5133


## Logistic regression summary by forecast window

This table compares the logistic regression baseline across each forecast horizon.


In [16]:
logistic_results_by_horizon = pd.concat(
    [
        logistic_1d_results,
        logistic_2d_results,
        logistic_3d_results,
        logistic_4d_results,
        logistic_10d_results,
        logistic_20d_results,
    ],
    ignore_index=True,
)

horizon_order = ['1d', '2d', '3d', '4d', '10d', '20d']
logistic_results_by_horizon['forecast_horizon'] = pd.Categorical(
    logistic_results_by_horizon['forecast_horizon'],
    categories=horizon_order,
    ordered=True,
)

logistic_results_by_horizon.sort_values('forecast_horizon').reset_index(drop=True)


,model,forecast_horizon,samples,up_rate,accuracy,log_loss,auc
0,Logistic regression,1d,24750,0.5255,0.5253,0.6922,0.4950
1,Logistic regression,2d,24651,0.5369,0.5302,0.6914,0.4942
2,Logistic regression,3d,24552,0.5427,0.5376,0.6903,0.4999
3,Logistic regression,4d,24453,0.5453,0.5424,0.6900,0.4952
4,Logistic regression,10d,23859,0.5647,0.5658,0.6847,0.5133
5,Logistic regression,20d,22869,0.5953,0.5952,0.6757,0.5133


## Random forest summary by forecast window

This table trains a random forest baseline for each forecast horizon using the same train/test data as the logistic regression baselines.


In [17]:
def get_random_forest_results(X_train, y_train, X_test, y_test, forecast_horizon):
    random_forest_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=30,
        random_state=42,
        n_jobs=-1,
    )
    random_forest_model.fit(X_train, y_train)

    y_prob = random_forest_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    random_forest_results = pd.DataFrame([
        {
            'model': 'Random forest',
            'forecast_horizon': f'{forecast_horizon}d',
            'samples': len(y_test),
            'up_rate': y_test.mean(),
            'accuracy': accuracy_score(y_test, y_pred),
            'log_loss': log_loss(y_test, y_prob, labels=[0, 1]),
            'auc': roc_auc_score(y_test, y_prob),
        },
    ])

    random_forest_results[metric_cols] = random_forest_results[metric_cols].round(4)
    return random_forest_model, random_forest_results


random_forest_1d_model, random_forest_1d_results = get_random_forest_results(X_train, y_train, X_test, y_test, 1)
random_forest_2d_model, random_forest_2d_results = get_random_forest_results(X_train_2d, y_train_2d, X_test_2d, y_test_2d, 2)
random_forest_3d_model, random_forest_3d_results = get_random_forest_results(X_train_3d, y_train_3d, X_test_3d, y_test_3d, 3)
random_forest_4d_model, random_forest_4d_results = get_random_forest_results(X_train_4d, y_train_4d, X_test_4d, y_test_4d, 4)
random_forest_10d_model, random_forest_10d_results = get_random_forest_results(X_train_10d, y_train_10d, X_test_10d, y_test_10d, 10)
random_forest_20d_model, random_forest_20d_results = get_random_forest_results(X_train_20d, y_train_20d, X_test_20d, y_test_20d, 20)

random_forest_results_by_horizon = pd.concat(
    [
        random_forest_1d_results,
        random_forest_2d_results,
        random_forest_3d_results,
        random_forest_4d_results,
        random_forest_10d_results,
        random_forest_20d_results,
    ],
    ignore_index=True,
)

random_forest_results_by_horizon['forecast_horizon'] = pd.Categorical(
    random_forest_results_by_horizon['forecast_horizon'],
    categories=horizon_order,
    ordered=True,
)

random_forest_results_by_horizon.sort_values('forecast_horizon').reset_index(drop=True)


,model,forecast_horizon,samples,up_rate,accuracy,log_loss,auc
0,Random forest,1d,24750,0.5255,0.5251,0.6918,0.5002
1,Random forest,2d,24651,0.5369,0.5323,0.6906,0.5076
2,Random forest,3d,24552,0.5427,0.5402,0.6894,0.5081
3,Random forest,4d,24453,0.5453,0.5426,0.6882,0.5271
4,Random forest,10d,23859,0.5647,0.5647,0.6847,0.5070
5,Random forest,20d,22869,0.5953,0.5949,0.6764,0.4900


This shows that Random Forest doesn't seem to provide any benefits over logistic regression.

## 20-Day Logistic Regression by Sector

This trains a separate 20-day logistic regression baseline for each sector group, then compares each sector model against its own sector-level naive majority baseline.


In [18]:
SECTOR_FORECAST_HORIZON = 20
sector_results = []
sector_logistic_models = {}

for sector, sector_tickers in TICKER_GROUPS.items():
    sector_ticker_features = {
        ticker: ticker_features[ticker]
        for ticker in sector_tickers
        if ticker in ticker_features
    }

    if not sector_ticker_features:
        continue

    sector_target_col = f'target_up_{SECTOR_FORECAST_HORIZON}d'
    sector_model_data = get_model_data(
        sector_ticker_features,
        LOGISTIC_BASELINE_FEATURES,
        SECTOR_FORECAST_HORIZON,
    )

    (
        sector_train_rows,
        sector_test_rows,
        sector_scaler,
        sector_X_train,
        sector_X_test,
        sector_y_train,
        sector_y_test,
    ) = get_train_test_data(
        sector_model_data,
        LOGISTIC_BASELINE_FEATURES,
        sector_target_col,
        TRAIN_END,
        TEST_START,
    )

    sector_naive_results = get_naive_baseline_results(sector_y_test, SECTOR_FORECAST_HORIZON)
    sector_logistic_model, sector_logistic_results = get_logistic_regression_results(
        sector_X_train,
        sector_y_train,
        sector_X_test,
        sector_y_test,
        SECTOR_FORECAST_HORIZON,
    )

    sector_naive_accuracy = sector_naive_results['accuracy'].iloc[0]
    sector_comparison = pd.concat([sector_naive_results, sector_logistic_results], ignore_index=True)
    sector_comparison.insert(0, 'sector', sector)
    sector_comparison.insert(1, 'tickers', len(sector_ticker_features))
    sector_comparison['naive_accuracy'] = sector_naive_accuracy
    sector_comparison['beats_naive'] = sector_comparison['accuracy'] > sector_naive_accuracy

    sector_results.append(sector_comparison)
    sector_logistic_models[sector] = sector_logistic_model

sector_logistic_20d_results = pd.concat(sector_results, ignore_index=True)
sector_logistic_20d_results = sector_logistic_20d_results.sort_values(
    ['sector', 'model']
).reset_index(drop=True)
sector_logistic_20d_results


,sector,tickers,model,forecast_horizon,samples,up_rate,predicted_class,accuracy,log_loss,auc,naive_accuracy,beats_naive
0,Consumer staples and discretionary,13,Logistic regression,20d,3003,0.5751,NaN,0.5691,0.6894,0.4921,0.5751,False
1,Consumer staples and discretionary,13,Naive majority baseline,20d,3003,0.5751,1.0,0.5751,0.6818,0.5000,0.5751,False
2,Energy,7,Logistic regression,20d,1617,0.4867,NaN,0.3865,0.7262,0.3552,0.5133,False
3,Energy,7,Naive majority baseline,20d,1617,0.4867,0.0,0.5133,0.6928,0.5000,0.5133,False
4,Financials,13,Logistic regression,20d,3003,0.6990,NaN,0.6990,0.6371,0.5327,0.6990,False
5,Financials,13,Naive majority baseline,20d,3003,0.6990,1.0,0.6990,0.6117,0.5000,0.6990,False
6,Healthcare,14,Logistic regression,20d,3234,0.5213,NaN,0.5090,0.7017,0.4759,0.5213,False
7,Healthcare,14,Naive majority baseline,20d,3234,0.5213,1.0,0.5213,0.6922,0.5000,0.5213,False
8,Industrials,11,Logistic regression,20d,2541,0.6021,NaN,0.6009,0.6724,0.5491,0.6021,False
9,Industrials,11,Naive majority baseline,20d,2541,0.6021,1.0,0.6021,0.6721,0.5000,0.6021,False


## Findings

This study shows that training the models by sector can help to improve results over a single pooled model.  The best results came from Utilities, Real Estate, Materials, and Industrials.